# LLM 蒸馏 — 怎么把大模型的能力「压缩」到小模型？

> **背景**：GPT-4 很强但太贵太慢。能不能把它的能力「蒸馏」到一个 7B 的小模型里，让 7B 模型也有接近 GPT-4 的水平？
>
> 本 Part 目标：**理解 LLM 蒸馏的三种主流方法，并掌握从 GPT-4 蒸馏到小模型的完整流程。**

核心思想一句话：**蒸馏 = 让大模型（Teacher）教小模型（Student），小模型学大模型的「思维方式」而不只是「标准答案」。**

> 📎 本 Part 是蒸馏的「全景指南」。关于 On-Policy Distillation（OPD）的深入分析，见 [支线 23：OPD](./23-opd.ipynb)。

In [1]:
import numpy as np

np.random.seed(42)

### 1. 蒸馏的本质：学「怎么想」而不只是「答案是什么」

```
普通 SFT（监督微调）:
  Teacher 输出: "巴黎"
  Student 学习: 输入"法国首都是？" → 输出"巴黎"
  问题: 只学了答案，没学推理过程

蒸馏:
  Teacher 输出: 每个词的概率分布 [巴黎:0.9, 伦敦:0.05, 柏林:0.03, ...]
  Student 学习: 不仅输出"巴黎"，还要让整个概率分布接近 Teacher
  好处: Student 学到了 Teacher 的「判断力」——知道"巴黎"最可能，"伦敦"也有可能但概率低
```

**为什么概率分布比答案更有价值？**

Teacher 说「巴黎 90%，伦敦 5%，柏林 3%」比只说「巴黎」多了两条信息：
1. 伦敦和柏林也是合理的（只是不太对）——这叫「暗知识」
2. 其他几百个城市概率接近 0——明确告诉 Student 哪些是错的

这就是 Hinton 2015 年提出知识蒸馏的核心洞察。

In [2]:
# 交互演示：硬标签 vs 软标签，用真实概率分布对比
print("=== 硬标签 vs 软标签 ===")
print()

cities = ["巴黎", "伦敦", "柏林", "罗马", "马德里", "东京", "北京", "悉尼"]
teacher_logits = np.array([5.0, 2.0, 1.0, 0.5, 0.1, -3.0, -4.0, -5.0])

# 硬标签 (SFT): one-hot
hard_labels = np.zeros(len(cities))
hard_labels[0] = 1.0

print("问题: 法国的首都是？")
print()
print("硬标签 (SFT):")
for city, prob in zip(cities[:5], hard_labels[:5]):
    bar = "█" * int(prob * 40)
    print(f"  {city}: {prob:.1%} {bar}")
print("  → Student 只知道「巴黎是对的」")
print()

# 软标签 (蒸馏): 概率分布
temperature = 3.0
scaled_logits = teacher_logits / temperature
soft_labels = np.exp(scaled_logits) / np.exp(scaled_logits).sum()

print("软标签 (蒸馏, T=3):")
for city, prob in zip(cities, soft_labels):
    bar = "█" * int(prob * 40)
    print(f"  {city}: {prob:.1%} {bar}")
print("  → Student 学到了:")
print("     1. 巴黎最对")
print("     2. 伦敦、柏林也是欧洲首都（相似性知识）")
print("     3. 东京、北京概率≈0（完全不相关）")
print()

# 量化信息量差异
hard_entropy = -np.sum(hard_labels * np.log(hard_labels + 1e-10))
soft_entropy = -np.sum(soft_labels * np.log(soft_labels + 1e-10))
print(f"硬标签信息熵: {hard_entropy:.2f} bits")
print(f"软标签信息熵: {soft_entropy:.2f} bits")
print(f"→ 软标签包含约 {soft_entropy:.1f} bits 信息，远多于硬标签的 {hard_entropy:.2f} bits！")

=== 硬标签 vs 软标签 ===

问题: 法国的首都是？

硬标签 (SFT):
  巴黎: 100.0% ████████████████████████████████████████
  伦敦: 0.0% 
  柏林: 0.0% 
  罗马: 0.0% 
  马德里: 0.0% 
  → Student 只知道「巴黎是对的」

软标签 (蒸馏, T=3):
  巴黎: 45.4% ██████████████████
  伦敦: 16.7% ██████
  柏林: 12.0% ████
  罗马: 10.1% ████
  马德里: 8.9% ███
  东京: 3.2% █
  北京: 2.3% 
  悉尼: 1.6% 
  → Student 学到了:
     1. 巴黎最对
     2. 伦敦、柏林也是欧洲首都（相似性知识）
     3. 东京、北京概率≈0（完全不相关）

硬标签信息熵: -0.00 bits
软标签信息熵: 1.62 bits
→ 软标签包含约 1.6 bits 信息，远多于硬标签的 -0.00 bits！


### 2. 方法一：Logit 蒸馏（最经典）

让 Student 的输出概率分布逼近 Teacher 的输出概率分布。

**Loss 公式**：

$$L = (1-\alpha) \cdot L_{CE}(S, y) + \alpha \cdot T^2 \cdot L_{KL}(S_T, T_T)$$

其中：
- $L_{CE}$：Student 和正确答案的交叉熵（保证基本正确）
- $L_{KL}$：Student 和 Teacher 概率分布的 KL 散度（学习暗知识）
- $T$：温度参数，越大 Teacher 的分布越「软」（暗知识越明显）
- $\alpha$：平衡两个 loss 的权重

**温度 T 的作用**：
```
T=1:  [0.90, 0.05, 0.03, 0.02]  ← 很尖锐，暗知识不明显
T=5:  [0.40, 0.25, 0.20, 0.15]  ← 软化了，暗知识浮现
T=20: [0.28, 0.26, 0.24, 0.22]  ← 太软了，变成均匀分布
```

T 太大 → 所有词概率差不多 → 没信息量
T 太小 → 和硬标签没区别 → 没暗知识
通常 T=3~10 比较合适。

In [3]:
# 演示温度对概率分布的影响
print("=== 温度 T 对软标签的影响 ===")
print()

logits = np.array([5.0, 2.0, 1.0, 0.5, 0.1, 0.01, 0.001, 0.0001])
labels = ["巴黎", "伦敦", "柏林", "罗马", "马德里", "维也纳", "布拉格", "华沙"]

for T in [1, 3, 10, 20]:
    scaled = logits / T
    probs = np.exp(scaled) / np.exp(scaled).sum()
    
    print(f"T={T:2d}: ", end="")
    for i in range(5):
        bar = "█" * int(probs[i] * 50)
        print(f"{labels[i]}:{probs[i]:.3f} {bar}  ", end="")
    print()

print()
print("T=1:  几乎只有巴黎 → 暗知识被掩盖")
print("T=3:  伦敦、柏林也有一定概率 → 暗知识浮现")
print("T=10: 分布更均匀 → 暗知识丰富但信号变弱")
print("T=20: 几乎均匀 → 信息量太少")

=== 温度 T 对软标签的影响 ===

T= 1: 巴黎:0.903 █████████████████████████████████████████████  伦敦:0.045 ██  柏林:0.017   罗马:0.010   马德里:0.007   
T= 3: 巴黎:0.382 ███████████████████  伦敦:0.141 ███████  柏林:0.101 █████  罗马:0.085 ████  马德里:0.075 ███  
T=10: 巴黎:0.182 █████████  伦敦:0.135 ██████  柏林:0.122 ██████  罗马:0.116 █████  马德里:0.112 █████  
T=20: 巴黎:0.152 ███████  伦敦:0.130 ██████  柏林:0.124 ██████  罗马:0.121 ██████  马德里:0.119 █████  

T=1:  几乎只有巴黎 → 暗知识被掩盖
T=3:  伦敦、柏林也有一定概率 → 暗知识浮现
T=10: 分布更均匀 → 暗知识丰富但信号变弱
T=20: 几乎均匀 → 信息量太少


### 3. 方法二：数据蒸馏（最实用）

Logit 蒸馏要求 Student 和 Teacher **词表相同**，这在 LLM 场景下几乎不可能（GPT-4 和 Qwen 词表不同）。

**数据蒸馏绕过了这个问题**：让 Teacher 生成训练数据，Student 在这些数据上做 SFT。

```
Step 1: 收集 prompts（从你的业务场景中）
  ["写一首关于春天的诗", "解释量子力学", "翻译: Hello → 中文", ...]

Step 2: Teacher 为每个 prompt 生成高质量回答
  GPT-4: "春天来了，万物复苏..."
  GPT-4: "量子力学是研究微观粒子..."

Step 3: 用 (prompt, teacher_answer) 对训练 Student
  Student 做标准 SFT，学习模仿 Teacher 的输出风格和质量
```

**优点**：不要求词表相同，任何 Teacher 可以教任何 Student。
**缺点**：只学到了「答案长什么样」，没学到「概率分布中的暗知识」。

**数据蒸馏的进阶技巧**：
- **多轮对话蒸馏**：Teacher 生成多轮对话，Student 学会对话节奏
- **CoT 蒸馏**：Teacher 生成带推理过程的答案，Student 学会推理
- **拒绝采样**：Teacher 生成多个答案，只保留最好的给 Student 学

In [4]:
# 模拟数据蒸馏流程
print("=== 数据蒸馏流程模拟 ===")
print()

prompts = [
    "解释什么是机器学习",
    "写一首关于秋天的五言诗",
    "Python 中 list 和 tuple 的区别",
]

# 模拟 Teacher (GPT-4) 生成
teacher_responses = [
    "机器学习是人工智能的一个分支，让计算机从数据中学习模式，而不需要显式编程。",
    "秋风扫落叶，霜降百花残。独坐寒窗下，思君衣可单。",
    "list 是可变的（可以增删改），tuple 是不可变的（创建后不能修改）。list 用 []，tuple 用 ()。",
]

print("生成训练数据:")
for i, (prompt, response) in enumerate(zip(prompts, teacher_responses)):
    print(f"\n--- 样本 {i+1} ---")
    print(f"User: {prompt}")
    print(f"Assistant: {response}")

print()
print(f"共生成 {len(prompts)} 条训练数据")
print("Student 在这些数据上做 SFT，学习模仿 Teacher 的风格。")
print()
print("实际项目中，通常需要 1万~10万 条这样的数据。")

=== 数据蒸馏流程模拟 ===

生成训练数据:

--- 样本 1 ---
User: 解释什么是机器学习
Assistant: 机器学习是人工智能的一个分支，让计算机从数据中学习模式，而不需要显式编程。

--- 样本 2 ---
User: 写一首关于秋天的五言诗
Assistant: 秋风扫落叶，霜降百花残。独坐寒窗下，思君衣可单。

--- 样本 3 ---
User: Python 中 list 和 tuple 的区别
Assistant: list 是可变的（可以增删改），tuple 是不可变的（创建后不能修改）。list 用 []，tuple 用 ()。

共生成 3 条训练数据
Student 在这些数据上做 SFT，学习模仿 Teacher 的风格。

实际项目中，通常需要 1万~10万 条这样的数据。


### 4. 方法三：特征蒸馏（进阶）

不仅学输出分布，还学中间层的表示。

```
Teacher (GPT-4, 96层):
  Layer 1 → Layer 2 → ... → Layer 48 → ... → Layer 96 → Output
                                      ↑
Student (7B, 32层):                  |  让 Student 第 16 层的输出
  Layer 1 → Layer 2 → ... → Layer 16 → ... → Layer 32 → Output  逼近 Teacher 第 48 层
```

**为什么有效？** 中间层包含了「怎么理解这句话」的信息，比最终输出更丰富。

**为什么少用？**
- 需要访问 Teacher 的内部表示（闭源模型不行）
- Teacher 和 Student 的维度不同，需要投影矩阵对齐
- 计算量大，显存消耗高

目前 LLM 蒸馏的主流还是**数据蒸馏**，特征蒸馏更多出现在视觉模型（如 ViT 蒸馏到 CNN）。

### 5. 实战：从 GPT-4 蒸馏一个 7B 模型

完整流程，每一步都可操作：

In [5]:
print("=== 实战：GPT-4 → 7B 蒸馏流程 ===")
print()

steps = [
    ("Step 1: 选基座模型", [
        "推荐: Qwen2.5-7B / Llama-3-8B / Mistral-7B",
        "要求: 基座模型本身有一定能力（不能太差）",
        "选 Instruct 版本（已经会遵循指令）",
    ]),
    ("Step 2: 收集 prompts", [
        "来源 1: 你的业务数据（用户真实问题）",
        "来源 2: 开源数据集（OpenHermes, ShareGPT, WildChat）",
        "来源 3: 自建——用另一个 LLM 生成多样化 prompt",
        "数量: 至少 5000 条，推荐 5万~10万 条",
    ]),
    ("Step 3: Teacher 生成回答", [
        "用 GPT-4 API 为每个 prompt 生成回答",
        "system prompt: '你是一个有帮助的助手，请详细、准确地回答。'",
        "temperature=0.7（保留一定多样性）",
        "成本: 5万条 × ~500 token/条 ≈ 2500万 token ≈ $250 (GPT-4o)",
    ]),
    ("Step 4: 数据清洗", [
        "去掉太短的回答（<20 token）",
        "去掉包含 '作为 AI' 等拒绝回答的",
        "去掉格式错乱的",
        "去重（相似度 > 0.9 的只保留一条）",
    ]),
    ("Step 5: SFT 训练", [
        "工具: LLaMA-Factory / Axolotl / Firefly",
        "格式: ChatML 或 ShareGPT 格式",
        "超参: lr=2e-5, epochs=3, batch_size=128",
        "硬件: 4×A100 (80G), 训练时间 ~6-12 小时",
    ]),
    ("Step 6: 评测", [
        "用 lm-eval 跑标准 benchmark（见支线 19）",
        "人工评估 100 条业务数据",
        "对比 Student 和 Teacher 的差距",
    ]),
]

for title, details in steps:
    print(title)
    for d in details:
        print(f"  {d}")
    print()

=== 实战：GPT-4 → 7B 蒸馏流程 ===

Step 1: 选基座模型
  推荐: Qwen2.5-7B / Llama-3-8B / Mistral-7B
  要求: 基座模型本身有一定能力（不能太差）
  选 Instruct 版本（已经会遵循指令）

Step 2: 收集 prompts
  来源 1: 你的业务数据（用户真实问题）
  来源 2: 开源数据集（OpenHermes, ShareGPT, WildChat）
  来源 3: 自建——用另一个 LLM 生成多样化 prompt
  数量: 至少 5000 条，推荐 5万~10万 条

Step 3: Teacher 生成回答
  用 GPT-4 API 为每个 prompt 生成回答
  system prompt: '你是一个有帮助的助手，请详细、准确地回答。'
  temperature=0.7（保留一定多样性）
  成本: 5万条 × ~500 token/条 ≈ 2500万 token ≈ $250 (GPT-4o)

Step 4: 数据清洗
  去掉太短的回答（<20 token）
  去掉包含 '作为 AI' 等拒绝回答的
  去掉格式错乱的
  去重（相似度 > 0.9 的只保留一条）

Step 5: SFT 训练
  工具: LLaMA-Factory / Axolotl / Firefly
  格式: ChatML 或 ShareGPT 格式
  超参: lr=2e-5, epochs=3, batch_size=128
  硬件: 4×A100 (80G), 训练时间 ~6-12 小时

Step 6: 评测
  用 lm-eval 跑标准 benchmark（见支线 19）
  人工评估 100 条业务数据
  对比 Student 和 Teacher 的差距



### 6. 蒸馏 vs OPD：什么时候用哪个？

| 维度 | 数据蒸馏 | OPD（On-Policy Distillation） |
|:---|:---|:---|
| **Teacher 参与时机** | 训练前（生成数据） | 训练中（实时打分） |
| **Student 训练数据** | Teacher 写的标准答案 | Student 自己写的答案 |
| **工程复杂度** | 低（就是 SFT） | 高（需要 rollout + 实时 Teacher） |
| **Exposure Bias** | 有（训练和推理 prefix 不一致） | 无（在真实 prefix 上训练） |
| **词表要求** | 无要求 | 需要对齐（或跨 tokenizer 蒸馏） |
| **成本** | 低（Teacher 只跑一次） | 高（Teacher 每次训练迭代都跑） |
| **适用场景** | 快速原型、预算有限 | 追求极致性能、有工程团队 |

**建议**：先用数据蒸馏快速出一个版本，效果好就上线；如果效果不够，再考虑 OPD。

In [6]:
# 蒸馏效果模拟
print("=== 蒸馏效果对比（模拟） ===")
print()

print("假设 Teacher 是 GPT-4，在 MMLU 上得分 86.4")
print()

models = [
    ("GPT-4 (Teacher)", 86.4, "基线"),
    ("7B 基座 (无蒸馏)", 64.2, "原始能力"),
    ("7B + 数据蒸馏", 72.8, "+8.6 分"),
    ("7B + 数据蒸馏 + CoT", 75.1, "+10.9 分"),
    ("7B + OPD", 77.3, "+13.1 分（但成本高 10 倍）"),
]

print(f"{'模型':<25s} {'MMLU':>8s} {'提升':>8s}")
print("-" * 45)
for name, score, note in models:
    improvement = score - 64.2
    print(f"{name:<25s} {score:>6.1f}  {improvement:>+6.1f}  ({note})")

print()
print("结论：数据蒸馏性价比最高，OPD 效果最好但成本高。")
print("实际项目中，数据蒸馏 + CoT 蒸馏是最常用的组合。")

=== 蒸馏效果对比（模拟） ===

假设 Teacher 是 GPT-4，在 MMLU 上得分 86.4

模型                            MMLU       提升
---------------------------------------------
GPT-4 (Teacher)             86.4   +22.2  (基线)
7B 基座 (无蒸馏)                 64.2    +0.0  (原始能力)
7B + 数据蒸馏                   72.8    +8.6  (+8.6 分)
7B + 数据蒸馏 + CoT             75.1   +10.9  (+10.9 分)
7B + OPD                    77.3   +13.1  (+13.1 分（但成本高 10 倍）)

结论：数据蒸馏性价比最高，OPD 效果最好但成本高。
实际项目中，数据蒸馏 + CoT 蒸馏是最常用的组合。


### 7. 蒸馏的常见问题

| 问题 | 答案 |
|:---|:---|
| **Student 能超过 Teacher 吗？** | 理论上不能（上限是 Teacher），但实践中如果 Student 有额外训练数据，可能在某些任务上超过 |
| **蒸馏会丢失什么？** | 创造性、长尾知识、复杂推理——这些是 Teacher 的「暗知识」中最难蒸馏的部分 |
| **需要多少数据？** | 最少 5000 条，推荐 5 万条以上。数据质量 > 数量 |
| **Teacher 和 Student 词表不同怎么办？** | 用数据蒸馏（方法二），不要求词表相同 |
| **蒸馏后还需要 RLHF 吗？** | 看场景。如果 Teacher 已经对齐过，蒸馏出的 Student 通常也对齐了 |
| **多 Teacher 蒸馏可行吗？** | 可以。GPT-4 教推理 + Claude 教写作 + Gemini 教多模态 → 综合能力更强 |

---

## LLM 蒸馏小结

1. ✅ **蒸馏本质**：学 Teacher 的概率分布（暗知识），而不只是标准答案
2. ✅ **Logit 蒸馏**：让 Student 的输出分布逼近 Teacher，需要同词表
3. ✅ **数据蒸馏**：Teacher 生成训练数据，Student 做 SFT，最实用
4. ✅ **特征蒸馏**：学中间层表示，效果好但工程复杂
5. ✅ **实战流程**：选基座 → 收集 prompt → Teacher 生成 → 清洗 → SFT → 评测
6. ✅ **蒸馏 vs OPD**：数据蒸馏性价比高，OPD 效果好但成本高
7. ✅ **CoT 蒸馏**：让 Teacher 生成带推理过程的答案，Student 学会推理

**一句话总结**：蒸馏 = 让大模型当老师，小模型当学生。
最实用的方法是数据蒸馏——让 GPT-4 写 5 万条回答，小模型照着学。